# Improved Clustering — Drop-in Replacement Cells

Replace the following cells in bagian1_banking_hints.ipynb:
- **Cell 18** → STEP A: Feature prep (drop collinear Monetary, add Monetary_cv)
- **Cell 18 (k-search loop)** → STEP B: Dual-metric k selection (Silhouette + Calinski-Harabasz)
- **Cell 19** → STEP C: Final model training
- **Cell 22** → STEP D: Auto-labeling from cluster centroids
- **Cell 23** → STEP E: Radar chart profile visualization

All other cells (data loading, preprocessing, RFM aggregation) are unchanged.

---
## STEP A — Feature Preparation
**What changed vs original:**
- Dropped `Monetary` (total sum) — it equals `Monetary_mean` × `Frequency`, so it's redundant.
  For the 75%+ of customers with Frequency=1, Monetary = Monetary_mean = Monetary_max: three
  columns saying the same thing gave KMeans three 'votes' for a signal it already had twice.
- Added `Monetary_cv` (coefficient of variation) — captures *spending erraticism*, which is
  informative only for repeat customers and acts as a differentiator inside that minority group.
- Kept everything else the same so the RFM aggregation cell above is untouched.

In [1]:
# ── STEP A: Feature selection & scaling ─────────────────────────────────────
# Run this cell INSTEAD of the old feature-prep cell (the one that defined
# CLUSTER_FEATURES as 11 items). Everything above (RFM aggregation) is unchanged.

import numpy as np
from numpy import log1p
from sklearn.preprocessing import StandardScaler

# KEY CHANGE: 'Monetary' (sum) removed — collinear with Monetary_mean when Frequency=1.
# 'Monetary_cv' added — captures spending consistency, differentiates repeat customers.
CLUSTER_FEATURES = [
    'Recency',          # days since last transaction
    'Frequency',        # unique transaction count
    'Monetary_mean',    # avg ticket size  (replaces raw Monetary sum)
    'Monetary_max',     # largest single transaction
    'Monetary_cv',      # NEW: spending erraticism (std/mean) — 0 for single-txn customers
    'Avg_Balance',      # account wealth (independent of spending)
    'Weekend_ratio',    # behavioral: weekday vs weekend preference
    'Hour_mean',        # behavioral: time-of-day preference
    'Active_months',    # seasonality / spread over calendar
    'Txn_span',         # loyalty tenure in days
    'Balance_to_spend', # wealth-to-spend ratio (saver vs spender)
]

LOG_FEATURES = [
    'Recency', 'Frequency', 'Monetary_mean', 'Monetary_max',
    'Avg_Balance', 'Txn_span', 'Balance_to_spend',
]

feat_df = rfm[CLUSTER_FEATURES].copy()
feat_df[LOG_FEATURES] = feat_df[LOG_FEATURES].apply(log1p)
feat_df = feat_df.fillna(0)   # handles NaN from missing CustAccountBalance / Monetary_cv

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(feat_df)

print(f'Feature matrix shape: {rfm_scaled.shape}')
print(f'Features ({len(CLUSTER_FEATURES)}): {CLUSTER_FEATURES}')

# Quick collinearity check — log Pearson correlation on log-transformed data
import pandas as pd
corr_check = pd.DataFrame(rfm_scaled, columns=CLUSTER_FEATURES).corr().round(2)
high_corr = [
    f'{a}↔{b}: {corr_check.loc[a,b]:.2f}'
    for i, a in enumerate(CLUSTER_FEATURES)
    for b in CLUSTER_FEATURES[i+1:]
    if abs(corr_check.loc[a,b]) > 0.80
]
print('\nHigh-correlation pairs (|r| > 0.80):',
      high_corr if high_corr else 'None — feature set is clean ✓')

NameError: name 'rfm' is not defined

---
## STEP B — Dual-metric K Selection
**What changed vs original:**
- Added **Calinski-Harabasz (CH) score** alongside Silhouette. CH penalises fragmentation
  of tight groups, so if k=4 is forced when k=2 is natural, CH will drop while Silhouette
  may still look acceptable. Only if *both* metrics agree is k genuinely good.
- A combined rank score picks the k that ranks highest on average across both metrics.
- The plot shows a third panel with that combined rank so the best k is unambiguous.

In [ ]:
# ── STEP B: Dual-metric K selection (Silhouette + Calinski-Harabasz) ─────────
import cupy as cp
import matplotlib.pyplot as plt
from cuml.cluster import KMeans as cuKMeans
from cuml.metrics.cluster import silhouette_score as cu_silhouette_score
from sklearn.metrics import calinski_harabasz_score   # CPU — fast on 11-dim array

X_gpu  = cp.asarray(rfm_scaled)
X_cpu  = rfm_scaled  # for Calinski-Harabasz (sklearn, CPU)

K_range    = range(2, 9)
inertias   = []
silhouettes = []
ch_scores  = []

for k in K_range:
    km     = cuKMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_gpu)
    labels_cpu = cp.asnumpy(labels)

    inertias.append(float(km.inertia_))
    silhouettes.append(float(cu_silhouette_score(X_gpu, labels)))
    ch_scores.append(calinski_harabasz_score(X_cpu, labels_cpu))

# ── Combined rank: lower rank = better (rank 1 = best) ──────────────────────
import numpy as np
sil_ranks = np.argsort(np.argsort(silhouettes)[::-1]) + 1  # rank 1 = highest silhouette
ch_ranks  = np.argsort(np.argsort(ch_scores)[::-1]) + 1    # rank 1 = highest CH
combined  = (sil_ranks + ch_ranks) / 2                      # lower is better
best_k    = list(K_range)[int(np.argmin(combined))]

print('k  | Silhouette | Calinski-Harabasz | Sil-rank | CH-rank | Combined')
print('-' * 70)
for i, k in enumerate(K_range):
    marker = '  ← BEST' if k == best_k else ''
    print(f'{k:2d} | {silhouettes[i]:10.4f} | {ch_scores[i]:17.1f} | {sil_ranks[i]:8d} | {ch_ranks[i]:7d} | {combined[i]:8.1f}{marker}')

# ── Plot all three panels ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

k_list = list(K_range)

axes[0].plot(k_list, inertias, 'bo-', linewidth=2)
axes[0].set_title('Elbow Method (Inertia)', fontweight='bold')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')

axes[1].plot(k_list, silhouettes, 'ro-', linewidth=2, label='Silhouette')
ax2 = axes[1].twinx()
ax2.plot(k_list, ch_scores, 'gs--', linewidth=1.5, label='CH Score')
axes[1].set_title('Silhouette (left) + Calinski-Harabasz (right)', fontweight='bold')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score', color='red')
ax2.set_ylabel('Calinski-Harabasz', color='green')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

axes[2].bar(k_list, combined, color=['gold' if k == best_k else 'steelblue' for k in K_range])
axes[2].set_title(f'Combined Rank Score (lower = better)\nBest k = {best_k}', fontweight='bold')
axes[2].set_xlabel('k'); axes[2].set_ylabel('Avg Rank (lower = better)')
axes[2].annotate(f'★ k={best_k}', xy=(best_k, combined[best_k - K_range.start]),
                 xytext=(0, 8), textcoords='offset points', ha='center',
                 fontweight='bold', color='darkred')

plt.tight_layout()
plt.savefig('elbow_silhouette_ch.png', dpi=150)
plt.show()

print(f'\n→ Best k by dual-metric ranking: {best_k}')

---
## STEP C — Final Model Training
**What changed vs original:**
- `BEST_K` is now set automatically from the dual-metric ranking above — no manual editing.
- Prints a clear summary of how each metric voted.

In [ ]:
# ── STEP C: Train final model with data-driven BEST_K ─────────────────────────
import time

# BEST_K is set automatically from the dual-metric ranking in STEP B.
# (In the original notebook it was hardcoded — now it's data-driven.)
BEST_K = best_k
print(f'Training final KMeans with k={BEST_K} (chosen by Silhouette + CH consensus)')
print(f'  Silhouette at k={BEST_K}: {silhouettes[BEST_K - K_range.start]:.4f}')
print(f'  CH Score   at k={BEST_K}: {ch_scores[BEST_K - K_range.start]:.1f}')

start = time.time()
kmeans_final = cuKMeans(n_clusters=BEST_K, random_state=42, n_init=10)
labels_gpu   = kmeans_final.fit_predict(X_gpu)
elapsed      = time.time() - start

rfm['Cluster'] = cp.asnumpy(labels_gpu)

final_sil = float(cu_silhouette_score(X_gpu, labels_gpu))
final_ch  = calinski_harabasz_score(X_cpu, cp.asnumpy(labels_gpu))

print(f'\nTraining time (GPU): {elapsed:.4f} s  |  rows: {len(rfm):,}')
print(f'Final Silhouette Score : {final_sil:.4f}')
print(f'Final Calinski-Harabasz: {final_ch:.1f}')

---
## STEP D — Auto-Labeling from Cluster Centroids
**What changed vs original:**
- Labels are derived from the actual cluster centroid values, not hardcoded guesses.
- Uses a simple scoring system: each cluster scores +1 for each dimension where
  its centroid is in the top half, −1 if in the bottom half. Clusters are then
  ranked from most-valuable to least-valuable (high F+M+Balance, low Recency).
- A fixed label hierarchy (Champions → Loyalists → At Risk → Dormant) is applied
  to those ranked clusters, so labels always reflect the actual data shape.

In [ ]:
# ── STEP D: Auto-label clusters from centroid positions ──────────────────────
import pandas as pd

profile = rfm.groupby('Cluster').agg(
    Recency       = ('Recency',       'mean'),
    Frequency     = ('Frequency',     'mean'),
    Monetary_mean = ('Monetary_mean', 'mean'),
    Avg_Balance   = ('Avg_Balance',   'mean'),
    Txn_span      = ('Txn_span',      'mean'),
    Count         = ('CustomerID',    'count'),
).round(2)

# Score each cluster: high Frequency/Monetary/Balance/Tenure = good; low Recency = good (recent)
medians = profile[['Recency','Frequency','Monetary_mean','Avg_Balance','Txn_span']].median()

profile['score'] = (
    - (profile['Recency']       > medians['Recency'])       # low recency is better
    + (profile['Frequency']     > medians['Frequency'])
    + (profile['Monetary_mean'] > medians['Monetary_mean'])
    + (profile['Avg_Balance']   > medians['Avg_Balance'])
    + (profile['Txn_span']      > medians['Txn_span'])
).astype(int)

# Rank clusters by score (descending).  Ties broken by Frequency then Monetary.
profile = profile.sort_values(['score','Frequency','Monetary_mean'], ascending=[False,False,False])

# Label hierarchy — maps rank position → business label
label_hierarchy = {
    0: 'Champions',         # highest engagement, highest value
    1: 'Loyalists',         # good value, moderate recency
    2: 'At Risk',           # declining — was active but slipping
    3: 'Dormant / Lost',    # low recency, low value, likely churned
    4: 'Low-Value Passive', # fallback for k > 4
    5: 'Occasional',
    6: 'Unknown',
}

cluster_to_label = {
    int(cluster): label_hierarchy[rank]
    for rank, cluster in enumerate(profile.index)
}
profile['Segment'] = [cluster_to_label[c] for c in profile.index]

rfm['Segment'] = rfm['Cluster'].map(cluster_to_label)

print('Cluster profiles (sorted by value score):')
print(profile[['Segment','Count','Recency','Frequency','Monetary_mean','Avg_Balance','score']].to_string())
print('\nSegment distribution:')
print(rfm['Segment'].value_counts().to_string())

---
## STEP E — Radar Chart Profile Visualization
**What changed vs original:**
- Replaced the bar chart with a **radar / spider chart** — far better for comparing
  multi-dimensional cluster profiles because you see the shape of each segment
  at a glance instead of reading bar heights one by one.
- Features are normalized to [0, 1] for the radar. Recency is inverted so that
  'high on the chart' always means 'more valuable' for every axis.
- A companion bar chart of cluster sizes is kept for quick headcount reference.

In [ ]:
# ── STEP E: Radar chart + cluster size bar ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.decomposition import PCA
import seaborn as sns
import pandas as pd

# ── 1. Radar chart ──────────────────────────────────────────────────────────
RADAR_FEATURES = [
    'Frequency', 'Monetary_mean', 'Avg_Balance',
    'Txn_span', 'Active_months', 'Weekend_ratio',
]
# Recency is excluded from radar because lower = better (confusing on radial axis);
# it's shown separately in the annotation.

radar_df = rfm.groupby('Segment')[RADAR_FEATURES].mean()

# Min-max normalise so all axes are 0–1 (higher = more of that feature)
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-9)

labels    = RADAR_FEATURES
N         = len(labels)
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles   += angles[:1]                 # close the polygon
palette   = sns.color_palette('Set2', n_colors=len(radar_norm))

fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                          gridspec_kw={'width_ratios': [2, 1]})
ax_r = fig.add_subplot(121, polar=True)
axes[0].remove()                        # replace left panel with polar

for (segment, row), color in zip(radar_norm.iterrows(), palette):
    values  = row.tolist() + row.tolist()[:1]
    ax_r.plot(angles, values, '-o', linewidth=2, color=color, label=segment, markersize=4)
    ax_r.fill(angles, values, alpha=0.12, color=color)

ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(
    ['Transaction\nFrequency', 'Avg Ticket\nSize', 'Account\nBalance',
     'Tenure\n(days)', 'Active\nMonths', 'Weekend\nRatio'],
    size=8.5
)
ax_r.set_yticks([0.25, 0.50, 0.75, 1.00])
ax_r.set_yticklabels(['25%', '50%', '75%', '100%'], size=7, color='grey')
ax_r.set_ylim(0, 1)
ax_r.set_title('Customer Segment Profiles\n(normalised 0–1; higher = more of that feature)',
               fontsize=11, fontweight='bold', pad=20)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

# Annotate with mean Recency (separately — lower is better)
recency_means = rfm.groupby('Segment')['Recency'].mean().round(1)
annot = '  Avg Recency (days — lower = more recent):\n'
for seg, val in recency_means.items():
    annot += f'    {seg}: {val}d\n'
fig.text(0.02, 0.01, annot, fontsize=8, va='bottom', color='#444')

# ── 2. Cluster size bar chart ───────────────────────────────────────────────
seg_counts = rfm['Segment'].value_counts().reindex(list(radar_norm.index))
bars = axes[1].bar(range(len(seg_counts)), seg_counts.values,
                   color=palette, edgecolor='white', linewidth=0.6)
axes[1].set_xticks(range(len(seg_counts)))
axes[1].set_xticklabels(seg_counts.index, rotation=25, ha='right', fontsize=9)
axes[1].set_title('Customers per Segment', fontweight='bold')
axes[1].set_ylabel('Customer Count')
for bar, count in zip(bars, seg_counts.values):
    pct = 100 * count / seg_counts.sum()
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + seg_counts.max() * 0.01,
                 f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('cluster_radar_and_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. PCA scatter (keep from original, updated segment labels) ─────────────
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)
ev = pca.explained_variance_ratio_

viz_df         = rfm.copy()
viz_df['PC1']  = rfm_pca[:, 0]
viz_df['PC2']  = rfm_pca[:, 1]
sample         = viz_df.sample(min(10_000, len(viz_df)), random_state=42)

fig2, ax = plt.subplots(figsize=(12, 7))
segments_sorted = rfm['Segment'].value_counts().index.tolist()

for segment, color in zip(segments_sorted, palette):
    pts = sample[sample['Segment'] == segment]
    ax.scatter(pts['PC1'], pts['PC2'], color=color, alpha=0.3, s=10, label=segment)
    cx, cy = pts['PC1'].mean(), pts['PC2'].mean()
    ax.scatter(cx, cy, color=color, s=200, marker='X', edgecolors='black', linewidths=1, zorder=6)
    ax.annotate(segment, xy=(cx, cy), xytext=(8, 6), textcoords='offset points',
                fontsize=8.5, fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.9))

ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% variance explained)', fontsize=10)
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% variance explained)', fontsize=10)
ax.set_title('Customer Segments — PCA 2D Projection (✕ = centroid)', fontsize=12, fontweight='bold')
ax.legend(title='Segment', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.axhline(0, color='grey', lw=0.5, alpha=0.4)
ax.axvline(0, color='grey', lw=0.5, alpha=0.4)
sns.despine()
plt.tight_layout()
plt.savefig('cluster_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('PCA variance captured:', f"{ev[:2].sum()*100:.1f}%")

---
## Updated Summary / Interpretation
Paste this block into the report narrative after the cluster profile table.

In [ ]:
# ── Print narrative-ready cluster summary ─────────────────────────────────────
print('=' * 70)
print('CLUSTER INTERPRETATION SUMMARY')
print('=' * 70)

full_profile = rfm.groupby('Segment').agg(
    Count         = ('CustomerID',    'count'),
    Recency_days  = ('Recency',       'mean'),
    Avg_Frequency = ('Frequency',     'mean'),
    Avg_Ticket    = ('Monetary_mean', 'mean'),
    Avg_Balance   = ('Avg_Balance',   'mean'),
    Tenure_days   = ('Txn_span',      'mean'),
).round(1)

total = full_profile['Count'].sum()
full_profile['Share_%'] = (full_profile['Count'] / total * 100).round(1)

print(full_profile.to_string())
print()

business_actions = {
    'Champions':        'Reward with loyalty perks, premium product upsell, referral programs.',
    'Loyalists':        'Cross-sell savings/investment products; maintain engagement cadence.',
    'At Risk':          'Trigger win-back campaigns; personalised incentives before churn.',
    'Dormant / Lost':   'Low-cost re-engagement (email/SMS); consider service fee waiver.',
    'Low-Value Passive':'Monitor only; route to digital self-service to reduce cost-to-serve.',
    'Occasional':       'Promote auto-debit or scheduled transactions to raise frequency.',
}

print('Recommended business actions per segment:')
for seg, action in business_actions.items():
    if seg in full_profile.index:
        n = int(full_profile.loc[seg, 'Count'])
        pct = full_profile.loc[seg, 'Share_%']
        print(f'  [{seg}]  n={n:,} ({pct}%)')
        print(f'    → {action}')